In [5]:
import pandas as pd
import numpy as ny
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

In [6]:
df=pd.read_csv("airquality_data (1).csv", encoding='cp1252', low_memory=False)

In [7]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435742 entries, 0 to 435741
Data columns (total 13 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   stn_code                     291665 non-null  object 
 1   sampling_date                435739 non-null  object 
 2   state                        435742 non-null  object 
 3   location                     435739 non-null  object 
 4   agency                       286261 non-null  object 
 5   type                         430349 non-null  object 
 6   so2                          401096 non-null  float64
 7   no2                          419509 non-null  float64
 8   rspm                         395520 non-null  float64
 9   spm                          198355 non-null  float64
 10  location_monitoring_station  408251 non-null  object 
 11  pm2_5                        9314 non-null    float64
 12  date                         435735 non-null  object 
dtyp

In [8]:
df['so2']=df['so2'].astype('float32')
df['no2']=df['no2'].astype('float32')
df['rspm']=df['rspm'].astype('float32')
df['spm']=df['spm'].astype('float32')
df['pm2_5']=df['pm2_5'].astype('float32')
df['date']=df['date'].astype('string')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435742 entries, 0 to 435741
Data columns (total 13 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   stn_code                     291665 non-null  object 
 1   sampling_date                435739 non-null  object 
 2   state                        435742 non-null  object 
 3   location                     435739 non-null  object 
 4   agency                       286261 non-null  object 
 5   type                         430349 non-null  object 
 6   so2                          401096 non-null  float32
 7   no2                          419509 non-null  float32
 8   rspm                         395520 non-null  float32
 9   spm                          198355 non-null  float32
 10  location_monitoring_station  408251 non-null  object 
 11  pm2_5                        9314 non-null    float32
 12  date                         435735 non-null  string 
dtyp

In [9]:
df=df.drop_duplicates()
print('shape after droping duplicates:',df.shape)

shape after droping duplicates: (435068, 13)


In [10]:
df.isna().sum()

stn_code                       144077
sampling_date                       3
state                               0
location                            3
agency                         149466
type                             5357
so2                             34632
no2                             16222
rspm                            40035
spm                            236908
location_monitoring_station     27303
pm2_5                          425754
date                                7
dtype: int64

In [11]:
percentage_missing=df.isnull().sum()*100/len(df)
percentage_missing.sort_values(ascending=False)

pm2_5                          97.859185
spm                            54.453097
agency                         34.354630
stn_code                       33.115973
rspm                            9.202010
so2                             7.960135
location_monitoring_station     6.275571
no2                             3.728613
type                            1.231302
date                            0.001609
sampling_date                   0.000690
location                        0.000690
state                           0.000000
dtype: float64

In [12]:
df=df.drop(['pm2_5','spm','agency','stn_code'] , axis=1)
df.head()

for col in df.columns:
  if df[col].dtype =='object' or df[col].dtype=='string':
    df[col]=df[col].fillna(df[col].mode()[0])
  else:
    df[col]=df[col].fillna(df[col].mean())

df.isna().sum()


sampling_date                  0
state                          0
location                       0
type                           0
so2                            0
no2                            0
rspm                           0
location_monitoring_station    0
date                           0
dtype: int64

In [13]:
subset1=df[['sampling_date','state']]
print(subset1.head())
subset2=df[['location','type']]
print(subset2.head())

        sampling_date           state
0  February - M021990  Andhra Pradesh
1  February - M021990  Andhra Pradesh
2  February - M021990  Andhra Pradesh
3     March - M031990  Andhra Pradesh
4     March - M031990  Andhra Pradesh
    location                                type
0  Hyderabad  Residential, Rural and other Areas
1  Hyderabad                     Industrial Area
2  Hyderabad  Residential, Rural and other Areas
3  Hyderabad  Residential, Rural and other Areas
4  Hyderabad                     Industrial Area


In [14]:
integrated_set = pd.concat([subset1,subset2],axis=1)
integrated_set.head()

,sampling_date,state,location,type
0,February - M021990,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas"
1,February - M021990,Andhra Pradesh,Hyderabad,Industrial Area
2,February - M021990,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas"
3,March - M031990,Andhra Pradesh,Hyderabad,"Residential, Rural and other Areas"
4,March - M031990,Andhra Pradesh,Hyderabad,Industrial Area


In [15]:
encoder=LabelEncoder()
for col in df.columns:
  df[col]=encoder.fit_transform(df[col])

df.head()


,sampling_date,state,location,type,so2,no2,rspm,location_monitoring_station,date
0,5358,0,114,6,446,1489,2030,757,213
1,5358,0,114,1,197,250,2030,757,213
2,5358,0,114,6,790,3096,2030,757,213
3,5421,0,114,6,823,1144,2030,757,214
4,5421,0,114,1,427,301,2030,757,214


In [16]:
def fix_outliers(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    column = column.clip(lower, upper)  # replace outliers
    return column

In [19]:
print("Before:\n", df.describe())

for col in df.columns:
    df[col] = fix_outliers(df[col])

print("After:\n", df.describe())

Before:
        sampling_date          state       location           type  \
count   435068.00000  435068.000000  435068.000000  435068.000000   
mean      2706.09162      18.724112     140.181769       4.288693   
std       1655.42713      10.089083      85.919163       2.221358   
min          0.00000       0.000000       0.000000       0.000000   
25%       1261.00000      12.000000      72.000000       2.000000   
50%       2682.00000      19.000000     134.000000       5.000000   
75%       4134.00000      27.000000     207.000000       6.000000   
max       5484.00000      36.000000     303.000000       9.000000   

                 so2            no2           rspm  \
count  435068.000000  435068.000000  435068.000000   
mean     1329.079470    2455.584058    1909.438589   
std       935.894926    1649.730131    1168.030342   
min         0.000000       0.000000       0.000000   
25%       502.000000    1038.000000    1035.000000   
50%      1389.000000    2176.000000    1870.0